# Pick out best coeffs

Nudge the best coefficients from the genetic algorithm results to check for better combinations near the current values. Calculate r-squared values to decide which coefficients are the best.

__Inputs:__

+ Best individuals for each generation of each run of the genetic algorithm.
+ MSOA-level admissions and population data for calculating admissions from resulting coefficients.
+ SSNAP-derived coefficients for converting scale factors into probability coefficients.

__Results:__

+ Calculated r-squared values for the fits of calculated to observed admission numbers.

__Method:__

For each run of the genetic algorithm, keep a copy of the best set of coefficients in the final generation.

Calculate the r-squared values associated with each set of coefficients.

Find all combos of coefficients nudged one or two significant figures either way. Then calculate r-squared for those new combos for each depriv quantile separately. Then group the best answer for each quantile to get the final set.

## Code setup

In [1]:
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

## Load data

In [2]:
df_best_gens = pl.read_csv(os.path.join('outputs', 'best_inds_deap.csv'))

In [3]:
df_best_gens

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,fitness,r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed00""",33.0,1.3,1.2,1.332,1.237,1.2,1.026,1.145,1.151,1.041,1.1,1.0,1.0,0.988,0.938,1.0,0.862,0.9,0.945,0.938,0.937,0.8,0.9,0.732,0.832,0.874,238.372,0.584365,0.502719,0.579147,0.616005,0.598245,0.606993
"""randomseed01""",33.0,1.1,1.3,1.305,1.3,1.231,1.1,1.1,1.1,1.023,1.1,1.0,1.0,1.0,1.0,1.0,0.969,0.9,0.9,0.892,0.9,0.837,0.849,0.8,0.8,0.894,238.507,0.575882,0.471584,0.579147,0.616005,0.583824,0.606993
"""randomseed02""",43.0,1.273,1.308,1.175,1.2,1.214,0.981,1.1,1.103,1.1,1.131,0.949,0.987,1.046,1.0,1.0,0.949,0.979,0.896,0.9,0.9,0.844,0.773,0.861,0.8,0.88,238.037,0.58383,0.485431,0.583466,0.616005,0.606683,0.606993
"""randomseed03""",28.0,1.171,1.3,1.327,1.186,1.3,1.1,1.198,1.1,1.1,1.1,1.042,1.0,1.0,1.0,0.971,0.881,0.9,0.894,1.0,0.9,0.8,0.771,0.808,0.7,0.9,239.49,0.585993,0.508469,0.583466,0.631482,0.605116,0.580367
"""randomseed04""",39.0,1.2,1.2,1.25,1.225,1.293,1.1,1.1,1.11,1.069,1.1,0.976,1.019,1.0,1.0,0.961,0.925,0.904,0.9,0.915,0.91,0.8,0.9,0.844,0.8,0.879,238.453,0.591769,0.508469,0.579147,0.631482,0.612783,0.606993
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""randomseed95""",35.0,1.329,1.313,1.248,1.175,1.222,1.165,1.1,1.1,1.015,1.097,0.91,0.988,0.943,1.0,1.0,0.866,0.9,0.9,0.971,0.9,0.7,0.841,0.9,0.812,0.9,238.173,0.583789,0.502719,0.569427,0.616005,0.605116,0.606993
"""randomseed96""",28.0,1.226,1.322,1.315,1.437,1.225,1.0,1.066,1.0,1.0,1.153,0.958,1.0,0.98,0.916,0.989,0.9,0.9,0.9,0.916,0.9,0.9,0.86,0.9,0.784,0.863,239.047,0.590145,0.506488,0.579147,0.631482,0.605116,0.608414
"""randomseed97""",46.0,1.211,1.217,1.217,1.2,1.299,1.027,1.129,1.1,1.106,1.1,1.027,1.0,0.97,0.922,1.0,0.937,0.986,0.9,0.9,0.9,0.799,0.8,0.9,0.876,0.859,238.515,0.587962,0.508469,0.583466,0.616005,0.606683,0.606993


## Load admissions data

In [4]:
path_to_msoa_stats = os.path.join('data', 'msoa_cleaned.csv')

df_stats = pl.read_csv(path_to_msoa_stats)

In [5]:
df_stats.head()

MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health,age_less65,age_65,age_70,age_75,age_over80,depriv_quantile_min,depriv_quantile_max
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64
"""Adur 001""",14.333333,16.924833,8815,"""E""",6799,1251,474,0.79763,0.146762,0.055608,"""E02006534""",0.0559,0.0528,0.0422,0.7872,0.062,8524,6939.168,492.7585,465.432,371.993,546.53,0.4,0.6
"""Adur 002""",7.333333,6.4704,7263,"""E""",5537,838,259,0.83464,0.126319,0.039041,"""E02006535""",0.0578,0.0774,0.0492,0.7467,0.0692,6634,5423.2821,419.8014,562.1562,357.3396,502.5996,0.8,1.0
"""Adur 003""",9.333333,13.7334,7354,"""E""",5820,969,311,0.819718,0.136479,0.043803,"""E02006536""",0.0609,0.0582,0.0421,0.7729,0.0661,7100,5683.9066,447.8586,428.0028,309.6034,486.0994,0.6,0.8
"""Adur 004""",21.0,26.199857,10582,"""E""",7872,1546,709,0.777328,0.152661,0.070011,"""E02006537""",0.0465,0.0438,0.0367,0.8091,0.0638,10127,8561.8962,492.063,463.4916,388.3594,675.1316,0.2,0.4
"""Adur 005""",13.666667,11.7948,9059,"""E""",7106,1081,339,0.833451,0.126789,0.039761,"""E02006538""",0.0597,0.067,0.0425,0.7643,0.0662,8526,6923.7937,540.8223,606.953,385.0075,599.7058,0.6,0.8


Pick out column names for the health and age proportions:

In [6]:
health_numbers = ['good_health', 'fair health', 'bad health']
props_health = ['prop_good_health', 'prop_fair health', 'prop_bad health']
props_age = [
    'age_less65_proportion', 'age_65_proportion', 'age_70_proportion',
    'age_75_proportion', 'age_over80_proportion'
]
age_numbers = [p.replace('_proportion', '') for p in props_age]

In [7]:
qmin_list = sorted(df_stats['depriv_quantile_min'].unique())

In [8]:
# Names of coeffs:
coeff_names = [f'{a}_q{str(round(q, 1)).replace(".", "")}' for q in qmin_list for a in age_numbers]

coeff_names

['age_less65_q00',
 'age_65_q00',
 'age_70_q00',
 'age_75_q00',
 'age_over80_q00',
 'age_less65_q02',
 'age_65_q02',
 'age_70_q02',
 'age_75_q02',
 'age_over80_q02',
 'age_less65_q04',
 'age_65_q04',
 'age_70_q04',
 'age_75_q04',
 'age_over80_q04',
 'age_less65_q06',
 'age_65_q06',
 'age_70_q06',
 'age_75_q06',
 'age_over80_q06',
 'age_less65_q08',
 'age_65_q08',
 'age_70_q08',
 'age_75_q08',
 'age_over80_q08']

Admissions data:

In [9]:
admissions_lists = []
x_lists = []

for qmin in qmin_list:
    df_stats_here = df_stats.filter(df_stats['depriv_quantile_min'] == qmin)
    # MSOA data in the same order as those coefficients:
    x_lists_here = [df_stats_here[a] for a in age_numbers]
    admissions_here = df_stats_here['admissions'].to_numpy().tolist()
    # Store:
    admissions_lists.append(admissions_here)
    x_lists.append(x_lists_here)

Starting SSNAP coefficients:

In [10]:
df_pop_admissions = pl.read_csv(os.path.join('outputs', 'ssnap_coeffs.csv'))

In [11]:
df_pop_admissions

Age Groups,population,prop_of_all_pop,count,prop_of_all_admissions,admissions_annual,admissions_annual_boost,prob_stroke_given_age
str,i64,f64,i64,f64,f64,f64,f64
"""Under 65""",45933245,0.816055,38827,0.231449,12942.33333,18737.66819,0.000408
"""65-69""",2796740,0.049687,15324,0.091347,5108.0,7395.26689,0.002644
"""70-74""",2779326,0.049378,21508,0.12821,7169.33333,10379.62674,0.003735
"""75-79""",1940686,0.034478,24150,0.143959,8050.0,11654.63948,0.006005
"""80 and over""",2836964,0.050402,67947,0.405035,22649.0,32790.7987,0.011558


Pick out SSNAP coefficients:

In [12]:
coeffs_ssnap = df_pop_admissions['prob_stroke_given_age'].to_numpy()

coeffs_ssnap

array([0.000408, 0.002644, 0.003735, 0.006005, 0.011558])

In [13]:
labels = ['less65', '65', '70', '75', 'over80']

# coeffs_ssnap_dict = dict(zip(df_pop_admissions['Age Groups'].to_numpy(), df_pop_admissions['prob_stroke_given_age'].to_numpy()))
coeffs_ssnap_dict = dict(zip(labels, df_pop_admissions['prob_stroke_given_age'].to_numpy()))

coeffs_ssnap_dict

{'less65': 0.000408,
 '65': 0.002644,
 '70': 0.003735,
 '75': 0.006005,
 'over80': 0.011558}

## Convert best scales to coeffs

In [14]:
for coeff in coeff_names:
    key = coeff.split('_')[1]
    new_data = df_best_gens[coeff] * coeffs_ssnap_dict[key]
    
    df_best_gens = df_best_gens.with_columns(pl.Series(coeff, new_data))

Round results:

In [15]:
labels = ['less65', '65', '70', '75', 'over80']
# round_dict = dict(zip(labels, [5, 4, 4, 4, 4]))
round_dict = dict(zip(labels, [4, 3, 3, 3, 3]))

for coeff in coeff_names:
    key = coeff.split('_')[1]
    new_data = np.round(df_best_gens[coeff], round_dict[key])
    
    df_best_gens = df_best_gens.with_columns(pl.Series(coeff, new_data))

In [16]:
df_best_gens

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,fitness,r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed00""",33.0,0.0005,0.003,0.005,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,238.372,0.584365,0.502719,0.579147,0.616005,0.598245,0.606993
"""randomseed01""",33.0,0.0004,0.003,0.005,0.008,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,238.507,0.575882,0.471584,0.579147,0.616005,0.583824,0.606993
"""randomseed02""",43.0,0.0005,0.003,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,238.037,0.58383,0.485431,0.583466,0.616005,0.606683,0.606993
"""randomseed03""",28.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.006,0.01,0.0003,0.002,0.003,0.004,0.01,239.49,0.585993,0.508469,0.583466,0.631482,0.605116,0.580367
"""randomseed04""",39.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,238.453,0.591769,0.508469,0.579147,0.631482,0.612783,0.606993
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""randomseed95""",35.0,0.0005,0.003,0.005,0.007,0.014,0.0005,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.006,0.01,0.0003,0.002,0.003,0.005,0.01,238.173,0.583789,0.502719,0.569427,0.616005,0.605116,0.606993
"""randomseed96""",28.0,0.0005,0.003,0.005,0.009,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.006,0.01,0.0004,0.002,0.003,0.005,0.01,239.047,0.590145,0.506488,0.579147,0.631482,0.605116,0.608414
"""randomseed97""",46.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,238.515,0.587962,0.508469,0.583466,0.616005,0.606683,0.606993


## Calculate r-squared

In [17]:
def predict_admissions(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    # yhat = (x_lists * coeffs.reshape(len(coeffs), 1)).sum(axis=0)
    yhat = sum(
        [x_lists[i] * coeffs[i] for i in range(len(coeffs))]
    )
    return yhat.to_numpy().tolist()

In [18]:
def calculate_rsquared(y, yhat):
    """This gives the same results as the sklearn built-in."""
    y = np.array(y)
    yhat = np.array(yhat)
    y_mean = np.mean(y)#.mean()
    ss_res = np.sum(((yhat - y)**2.0))#.sum()
    ss_tot = np.sum(((y - y_mean)**2.0))#.sum()
    if ss_tot != 0.0:
        rsq = 1.0 - ss_res / ss_tot
    else:
        rsq = np.NaN
    return rsq

In [19]:
def calculate_many_rsquared(coeffs, x_lists, coeffs_ssnap, admissions_lists):
    list_r2 = []
    predictions_lists = []
    for i in range(5):
        # Population numbers for areas in this quantile:
        x_lists_here = x_lists[i]
        # Separate prediction for each deprivation quantile:
        c = np.array(coeffs[(i*5):(i*5)+5]) #* np.array(coeffs_ssnap)
        yhat = predict_admissions(x_lists_here, c)
        predictions_lists.append(yhat)
        observed_here = admissions_lists[i]
        r2 = calculate_rsquared(observed_here, yhat)
        list_r2.append(r2)

    # Combine lists:
    observed_all = sum(admissions_lists, [])
    predicted_all = sum(predictions_lists, [])
    # Calculate r-squared:
    r2 = calculate_rsquared(observed_all, predicted_all)
    list_r2.append(r2)
    return list_r2

Check results:

In [20]:
df_best_gens.sort('r2_all', descending=True)

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,fitness,r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed24""",23.0,0.0005,0.004,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,238.421,0.593483,0.509119,0.583466,0.634746,0.612783,0.606993
"""randomseed08""",36.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.003,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,238.56,0.591779,0.508469,0.583466,0.631482,0.608507,0.606993
"""randomseed04""",39.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,238.453,0.591769,0.508469,0.579147,0.631482,0.612783,0.606993
"""randomseed17""",20.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.924,0.591412,0.508469,0.583466,0.631482,0.606683,0.606993
"""randomseed19""",34.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.009,0.591412,0.508469,0.583466,0.631482,0.606683,0.606993
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""randomseed65""",19.0,0.0005,0.004,0.005,0.008,0.014,0.0005,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.802,0.57744,0.505377,0.556782,0.616005,0.583824,0.606993
"""randomseed05""",19.0,0.0005,0.003,0.004,0.007,0.015,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,238.254,0.577355,0.502059,0.559312,0.616005,0.583824,0.606993
"""randomseed89""",25.0,0.0005,0.003,0.004,0.007,0.014,0.0005,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,238.894,0.576409,0.485431,0.569427,0.616005,0.583824,0.606993


In [21]:
# Pick out coeffs:
coeffs = df_best_gens.filter(df_best_gens['r2_all'] == df_best_gens['r2_all'].max())[coeff_names].to_numpy().flatten()
# Update some:
coeffs[coeff_names.index('age_75_q06')] = 0.006
coeffs[coeff_names.index('age_over80_q08')] = 0.011

list_r2 = calculate_many_rsquared(coeffs, x_lists, coeffs_ssnap, admissions_lists)
# R2 for each quantile in turn, then R2 of all.

list_r2

[0.5091191926826808,
 0.5834662232771566,
 0.6347460152228221,
 0.6160067681983439,
 0.624809983650576,
 0.5975473208650145]

In [22]:
coeffs.reshape(5, 5)

array([[0.0005, 0.004 , 0.005 , 0.007 , 0.014 ],
       [0.0004, 0.003 , 0.004 , 0.007 , 0.013 ],
       [0.0004, 0.002 , 0.004 , 0.006 , 0.012 ],
       [0.0004, 0.002 , 0.003 , 0.006 , 0.011 ],
       [0.0003, 0.002 , 0.003 , 0.005 , 0.011 ]])

In [23]:
r2_best = df_best_gens['r2_all'].max()

r2_best

0.5934830999955307

## Grid near best combo

Take the best coefficients. For each deprivation quantile, nudge the best coefficient one or two clicks either way and calculate the new fitness scores.

In [24]:
quantile_str_list = sorted(list(set([c.split('_')[-1] for c in coeff_names])))
age_str_list = ['age_less65', 'age_65', 'age_70', 'age_75', 'age_over80']  # sorted(list(set(['_'.join(c.split('_')[:-1]) for c in coeff_names])))

In [25]:
quantile_str_list

['q00', 'q02', 'q04', 'q06', 'q08']

In [26]:
age_str_list

['age_less65', 'age_65', 'age_70', 'age_75', 'age_over80']

In [27]:
import itertools

In [28]:

def calculate_new_combos_fitnesses(quantile_str, df):
    qi = quantile_str_list.index(quantile_str)
    
    best_coeff_cols = [f'{a}_{quantile_str}' for a in age_str_list]
    best_coeffs = df[best_coeff_cols].to_numpy().flatten()
    # New options:
    new_options = [
        [round(best_coeffs[0] + i*1e-4, 4) for i in range(-2, 3)],  # age less 65
        [round(best_coeffs[1] + i*1e-3, 3) for i in range(-2, 3)],  # age 65-69
        [round(best_coeffs[2] + i*1e-3, 3) for i in range(-2, 3)],  # age 70-74
        [round(best_coeffs[3] + i*1e-3, 3) for i in range(-2, 3)],  # age 75-79
        [round(best_coeffs[4] + i*1e-3, 3) for i in range(-2, 3)],  # age over 80
    ]
    # Generate all combinations of these new parameters:
    all_new_option_combos = list(itertools.product(*new_options))
    # Only keep combinations where coefficients increase with age band:
    all_new_option_combos = np.array(all_new_option_combos)[(np.diff(all_new_option_combos, axis=1) >= 0).all(axis=1)]
    
    # Calculate r-squared:
    dict_r2 = {}
    dict_r2_qs = [{}, {}, {}, {}, {}]
    
    list_r2 = []
    for flamingo, coeffs in enumerate(all_new_option_combos):
        # Pick out coeffs:
        # coeffs = df_best_gens.filter(df_best_gens['dir'] == results_dir)[coeff_names].to_numpy().flatten()
        
        # Population numbers for areas in this quantile:
        x_lists_here = x_lists[qi]
        # Separate prediction for each deprivation quantile:
        # c = np.array(coeffs[(i*5):(i*5)+5]) #* np.array(coeffs_ssnap)
        c = np.array(list(coeffs))
        yhat = predict_admissions(x_lists_here, c)
        observed_here = admissions_lists[qi]
        r2 = calculate_rsquared(observed_here, yhat)
        list_r2.append(r2)
    
    # Place new coeff combos into dataframe:
    df_new_combos = pl.DataFrame(all_new_option_combos, schema=best_coeff_cols, orient='row')
    df_new_combos = df_new_combos.with_columns(pl.Series(f'r2_{quantile_str}', list_r2))
    return df_new_combos

In [29]:
new_combo_df_dict = {}

for quantile_str in quantile_str_list:
    df_new_combos = calculate_new_combos_fitnesses(quantile_str, df_best_gens.filter(df_best_gens['r2_all'] == df_best_gens['r2_all'].max()))
    new_combo_df_dict[quantile_str] = df_new_combos

In [30]:
new_combo_df_dict.keys()

dict_keys(['q00', 'q02', 'q04', 'q06', 'q08'])

In [31]:
for quantile_str in list(new_combo_df_dict.keys()):
    display(df_best_gens.filter(df_best_gens['r2_all'] == df_best_gens['r2_all'].max())[[f'{a}_{quantile_str}' for a in age_str_list] + [f'r2_{quantile_str}', 'r2_all']])
    display(new_combo_df_dict[quantile_str].sort(f'r2_{quantile_str}', descending=True))

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,r2_q00,r2_all
f64,f64,f64,f64,f64,f64,f64
0.0005,0.004,0.005,0.007,0.014,0.509119,0.593483


age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,r2_q00
f64,f64,f64,f64,f64,f64
0.0004,0.006,0.006,0.006,0.014,0.510425
0.0005,0.005,0.005,0.005,0.014,0.510094
0.0004,0.006,0.006,0.007,0.013,0.510005
0.0004,0.005,0.006,0.006,0.015,0.509989
0.0004,0.005,0.005,0.006,0.016,0.509762
…,…,…,…,…,…
0.0007,0.006,0.007,0.009,0.015,-0.186458
0.0003,0.002,0.003,0.006,0.012,-0.191596
0.0007,0.006,0.007,0.008,0.016,-0.222319


age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,r2_q02,r2_all
f64,f64,f64,f64,f64,f64,f64
0.0004,0.003,0.004,0.007,0.013,0.583466,0.593483


age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,r2_q02
f64,f64,f64,f64,f64,f64
0.0004,0.003,0.004,0.006,0.014,0.583846
0.0004,0.004,0.004,0.006,0.013,0.583672
0.0004,0.003,0.003,0.007,0.014,0.583667
0.0004,0.003,0.003,0.006,0.015,0.583536
0.0004,0.003,0.004,0.007,0.013,0.583466
…,…,…,…,…,…
0.0002,0.001,0.003,0.005,0.011,-0.15007
0.0006,0.005,0.006,0.008,0.015,-0.174685
0.0002,0.001,0.002,0.006,0.011,-0.192834


age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,r2_q04,r2_all
f64,f64,f64,f64,f64,f64,f64
0.0004,0.002,0.004,0.006,0.012,0.634746,0.593483


age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,r2_q04
f64,f64,f64,f64,f64,f64
0.0003,0.002,0.002,0.007,0.014,0.646679
0.0003,0.002,0.003,0.006,0.014,0.646457
0.0003,0.001,0.003,0.007,0.014,0.646105
0.0003,0.003,0.003,0.004,0.014,0.645803
0.0003,0.002,0.004,0.004,0.014,0.645713
…,…,…,…,…,…
0.0006,0.004,0.005,0.008,0.014,-0.262324
0.0006,0.004,0.006,0.008,0.013,-0.2652
0.0006,0.003,0.006,0.008,0.014,-0.268051


age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,r2_q06,r2_all
f64,f64,f64,f64,f64,f64,f64
0.0004,0.002,0.003,0.005,0.011,0.612783,0.593483


age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,r2_q06
f64,f64,f64,f64,f64,f64
0.0003,0.002,0.002,0.006,0.013,0.624681
0.0003,0.001,0.003,0.006,0.013,0.624445
0.0003,0.001,0.002,0.007,0.013,0.624237
0.0003,0.002,0.003,0.005,0.013,0.623275
0.0003,0.003,0.003,0.003,0.013,0.622883
…,…,…,…,…,…
0.0006,0.003,0.005,0.007,0.013,-0.248632
0.0006,0.004,0.005,0.006,0.013,-0.294663
0.0002,0.001,0.001,0.004,0.009,-0.296746


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_q08,r2_all
f64,f64,f64,f64,f64,f64,f64
0.0003,0.002,0.003,0.005,0.01,0.606993,0.593483


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_q08
f64,f64,f64,f64,f64,f64
0.0001,0.003,0.004,0.004,0.012,0.634189
0.0002,0.001,0.004,0.005,0.012,0.633741
0.0002,0.002,0.003,0.005,0.012,0.633663
0.0001,0.002,0.003,0.007,0.012,0.633617
0.0001,0.003,0.003,0.006,0.012,0.633616
…,…,…,…,…,…
0.0001,0.001,0.001,0.003,0.009,-0.383149
0.0005,0.004,0.005,0.007,0.012,-0.385835
0.0001,0.001,0.002,0.003,0.008,-0.403286


In [32]:
for quantile_str in list(new_combo_df_dict.keys()):
    # display(df_best_gens.filter(df_best_gens['r2_all'] == df_best_gens['r2_all'].max())[[f'{a}_{quantile_str}' for a in age_str_list] + [f'r2_{quantile_str}', 'r2_all']])
    display(new_combo_df_dict[quantile_str].sort(f'r2_{quantile_str}', descending=True)[0])

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,r2_q00
f64,f64,f64,f64,f64,f64
0.0004,0.006,0.006,0.006,0.014,0.510425


age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,r2_q02
f64,f64,f64,f64,f64,f64
0.0004,0.003,0.004,0.006,0.014,0.583846


age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,r2_q04
f64,f64,f64,f64,f64,f64
0.0003,0.002,0.002,0.007,0.014,0.646679


age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,r2_q06
f64,f64,f64,f64,f64,f64
0.0003,0.002,0.002,0.006,0.013,0.624681


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_q08
f64,f64,f64,f64,f64,f64
0.0001,0.003,0.004,0.004,0.012,0.634189


In [33]:
best_combos = []
best_combo_cols = []

for q, quantile_str in enumerate(list(new_combo_df_dict.keys())):
    # display(df_best_gens.filter(df_best_gens['r2_all'] == df_best_gens['r2_all'].max())[[f'{a}_{quantile_str}' for a in age_str_list] + [f'r2_{quantile_str}', 'r2_all']])
    df = new_combo_df_dict[quantile_str].sort(f'r2_{quantile_str}', descending=True)
    df = df[0] if q < 4 else df[3]
    cols = [c for c in df.columns if c.startswith('age')]
    best_combo_cols += cols
    best_combos += list(df[cols].to_numpy().flatten())

In [34]:
best_combo_cols

['age_less65_q00',
 'age_65_q00',
 'age_70_q00',
 'age_75_q00',
 'age_over80_q00',
 'age_less65_q02',
 'age_65_q02',
 'age_70_q02',
 'age_75_q02',
 'age_over80_q02',
 'age_less65_q04',
 'age_65_q04',
 'age_70_q04',
 'age_75_q04',
 'age_over80_q04',
 'age_less65_q06',
 'age_65_q06',
 'age_70_q06',
 'age_75_q06',
 'age_over80_q06',
 'age_less65_q08',
 'age_65_q08',
 'age_70_q08',
 'age_75_q08',
 'age_over80_q08']

Check that it meets the requirements of increasing with age and deprivation:

In [37]:
np.array(best_combos).reshape(5, 5)

array([[0.0004, 0.006 , 0.006 , 0.006 , 0.014 ],
       [0.0004, 0.003 , 0.004 , 0.006 , 0.014 ],
       [0.0003, 0.002 , 0.002 , 0.007 , 0.014 ],
       [0.0003, 0.002 , 0.002 , 0.006 , 0.013 ],
       [0.0001, 0.002 , 0.003 , 0.007 , 0.012 ]])

Recalculate total r-squared:

In [38]:
list_r2 = calculate_many_rsquared(best_combos, x_lists, coeffs_ssnap, admissions_lists)
# R2 for each quantile in turn, then R2 of all.

list_r2

[0.5104249832646602,
 0.5838457808004065,
 0.6466793454130462,
 0.6246806345257032,
 0.6336174528123418,
 0.6039506773008061]

### Jiggle all 100 fits

In [39]:
new_combo_100_df_dict = {}

for quantile_str in quantile_str_list:
    print(quantile_str)
    r2_col = f'r2_{quantile_str}'
    r2_best_q = df_best_gens[r2_col].max() 
    for i in range(len(df_best_gens)):
        print(f'{i+1:3d} out of {len(df_best_gens)}', end='\r')
        df = calculate_new_combos_fitnesses(quantile_str, df_best_gens[i])
        # Only keep decent combos:
        df = df.filter(df[r2_col] >= (0.95 * r2_best_q))
        if i == 0:
            df_new_combos = df
        else:
            df_new_combos = pl.concat((df_new_combos, df), how='vertical')
            # Remove duplicate columns:
            df_new_combos = df_new_combos.unique()
    
    new_combo_100_df_dict[quantile_str] = df_new_combos

q00
q02 out of 100
q04 out of 100
q06 out of 100
q08 out of 100


In [40]:
for quantile_str in list(new_combo_df_dict.keys()):
    # display(df_best_gens.filter(df_best_gens['r2_all'] == df_best_gens['r2_all'].max())[[f'{a}_{quantile_str}' for a in age_str_list] + [f'r2_{quantile_str}', 'r2_all']])
    display(new_combo_100_df_dict[quantile_str].sort(f'r2_{quantile_str}', descending=True))

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,r2_q00
f64,f64,f64,f64,f64,f64
0.0004,0.006,0.006,0.006,0.014,0.510425
0.0005,0.005,0.005,0.005,0.014,0.510094
0.0004,0.006,0.006,0.007,0.013,0.510005
0.0004,0.005,0.006,0.006,0.015,0.509989
0.0004,0.005,0.005,0.006,0.016,0.509762
…,…,…,…,…,…
0.0004,0.005,0.005,0.01,0.015,0.483708
0.0004,0.004,0.007,0.01,0.014,0.483701
0.0003,0.003,0.007,0.007,0.016,0.483695


age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,r2_q02
f64,f64,f64,f64,f64,f64
0.0004,0.004,0.004,0.004,0.014,0.584399
0.0004,0.003,0.004,0.004,0.015,0.584127
0.0004,0.003,0.004,0.006,0.014,0.583846
0.0004,0.004,0.004,0.006,0.013,0.583672
0.0004,0.003,0.003,0.007,0.014,0.583667
…,…,…,…,…,…
0.0004,0.001,0.005,0.007,0.016,0.554771
0.0002,0.003,0.004,0.006,0.016,0.554734
0.0006,0.001,0.005,0.007,0.011,0.554707


age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,r2_q04
f64,f64,f64,f64,f64,f64
0.0003,0.002,0.002,0.007,0.014,0.646679
0.0003,0.002,0.003,0.006,0.014,0.646457
0.0003,0.001,0.003,0.007,0.014,0.646105
0.0003,0.003,0.003,0.004,0.014,0.645803
0.0003,0.002,0.004,0.004,0.014,0.645713
…,…,…,…,…,…
0.0004,0.003,0.004,0.006,0.009,0.603425
0.0005,0.004,0.004,0.006,0.009,0.603402
0.0005,0.002,0.003,0.006,0.01,0.603252


age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,r2_q06
f64,f64,f64,f64,f64,f64
0.0003,0.002,0.002,0.006,0.013,0.624681
0.0003,0.001,0.003,0.006,0.013,0.624445
0.0003,0.001,0.002,0.007,0.013,0.624237
0.0003,0.002,0.003,0.005,0.013,0.623275
0.0003,0.003,0.003,0.003,0.013,0.622883
…,…,…,…,…,…
0.0005,0.003,0.005,0.005,0.008,0.585325
0.0003,0.001,0.004,0.008,0.009,0.58531
0.0003,0.002,0.005,0.007,0.008,0.58531


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_q08
f64,f64,f64,f64,f64,f64
0.0001,0.003,0.003,0.004,0.013,0.636718
0.0001,0.002,0.004,0.004,0.013,0.636306
0.0002,0.002,0.002,0.005,0.013,0.635773
0.0002,0.001,0.003,0.005,0.013,0.635698
0.0002,0.002,0.003,0.004,0.013,0.635487
…,…,…,…,…,…
0.0002,0.003,0.004,0.005,0.012,0.594071
0.0004,0.001,0.005,0.006,0.009,0.593988
0.0003,0.001,0.002,0.007,0.013,0.593886


In [41]:
for quantile_str in list(new_combo_100_df_dict.keys()):
    display(df_best_gens.filter(df_best_gens['r2_all'] == df_best_gens['r2_all'].max())[[f'{a}_{quantile_str}' for a in age_str_list] + [f'r2_{quantile_str}', 'r2_all']])
    display(new_combo_100_df_dict[quantile_str].sort(f'r2_{quantile_str}', descending=True)[0])

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,r2_q00,r2_all
f64,f64,f64,f64,f64,f64,f64
0.0005,0.004,0.005,0.007,0.014,0.509119,0.593483


age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,r2_q00
f64,f64,f64,f64,f64,f64
0.0004,0.006,0.006,0.006,0.014,0.510425


age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,r2_q02,r2_all
f64,f64,f64,f64,f64,f64,f64
0.0004,0.003,0.004,0.007,0.013,0.583466,0.593483


age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,r2_q02
f64,f64,f64,f64,f64,f64
0.0004,0.004,0.004,0.004,0.014,0.584399


age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,r2_q04,r2_all
f64,f64,f64,f64,f64,f64,f64
0.0004,0.002,0.004,0.006,0.012,0.634746,0.593483


age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,r2_q04
f64,f64,f64,f64,f64,f64
0.0003,0.002,0.002,0.007,0.014,0.646679


age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,r2_q06,r2_all
f64,f64,f64,f64,f64,f64,f64
0.0004,0.002,0.003,0.005,0.011,0.612783,0.593483


age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,r2_q06
f64,f64,f64,f64,f64,f64
0.0003,0.002,0.002,0.006,0.013,0.624681


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_q08,r2_all
f64,f64,f64,f64,f64,f64,f64
0.0003,0.002,0.003,0.005,0.01,0.606993,0.593483


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_q08
f64,f64,f64,f64,f64,f64
0.0001,0.003,0.003,0.004,0.013,0.636718


In [42]:
best_100_combos = []
best_100_combo_cols = []

for q, quantile_str in enumerate(list(new_combo_100_df_dict.keys())):
    # display(df_best_gens.filter(df_best_gens['r2_all'] == df_best_gens['r2_all'].max())[[f'{a}_{quantile_str}' for a in age_str_list] + [f'r2_{quantile_str}', 'r2_all']])
    df = new_combo_100_df_dict[quantile_str].sort(f'r2_{quantile_str}', descending=True)
    df = df[0]# if q < 4 else df[3]
    cols = [c for c in df.columns if c.startswith('age')]
    best_100_combo_cols += cols
    best_100_combos += list(df[cols].to_numpy().flatten())

In [43]:
best_100_combo_cols

['age_less65_q00',
 'age_65_q00',
 'age_70_q00',
 'age_75_q00',
 'age_over80_q00',
 'age_less65_q02',
 'age_65_q02',
 'age_70_q02',
 'age_75_q02',
 'age_over80_q02',
 'age_less65_q04',
 'age_65_q04',
 'age_70_q04',
 'age_75_q04',
 'age_over80_q04',
 'age_less65_q06',
 'age_65_q06',
 'age_70_q06',
 'age_75_q06',
 'age_over80_q06',
 'age_less65_q08',
 'age_65_q08',
 'age_70_q08',
 'age_75_q08',
 'age_over80_q08']

Check that it meets the requirements of increasing with age and deprivation:

In [44]:
np.array(best_100_combos).reshape(5, 5)

array([[0.0004, 0.006 , 0.006 , 0.006 , 0.014 ],
       [0.0004, 0.004 , 0.004 , 0.004 , 0.014 ],
       [0.0003, 0.002 , 0.002 , 0.007 , 0.014 ],
       [0.0003, 0.002 , 0.002 , 0.006 , 0.013 ],
       [0.0001, 0.003 , 0.003 , 0.004 , 0.013 ]])

They don't, so instead try all combos of fixing one of these best combos and then picking the other ones that work with it.

In [45]:
def pick_same_increase(combo, all_combos):
    masks = [(all_combos[:, p] >= combo[p]) for p in range(len(combo))]
    masks = np.vstack(masks).T
    valid_mask = masks.all(axis=1)
    pick = np.where(valid_mask == True)[0][0]
    return all_combos[pick]

def pick_same_decrease(combo, all_combos):
    masks = [(all_combos[:, p] <= combo[p]) for p in range(len(combo))]
    masks = np.vstack(masks).T
    valid_mask = masks.all(axis=1)
    pick = np.where(valid_mask == True)[0][0]
    return all_combos[pick]

In [46]:
best_picked_combos = []

for q, quantile_str in enumerate(list(new_combo_100_df_dict.keys())):
    # display(df_best_gens.filter(df_best_gens['r2_all'] == df_best_gens['r2_all'].max())[[f'{a}_{quantile_str}' for a in age_str_list] + [f'r2_{quantile_str}', 'r2_all']])
    fixed_combo = new_combo_100_df_dict[quantile_str].sort(f'r2_{quantile_str}', descending=True)[0].to_numpy().flatten()[:-1]

    # Set up coeff storage:
    best_combo = [[]] * 5
    best_combo[q] = list(fixed_combo)
    # Find the combos where values should be less than or same as the
    # starting values:
    for p in range(q+1, 5):
        p_str = list(new_combo_100_df_dict.keys())[p]
        arr = new_combo_100_df_dict[p_str].sort(f'r2_{p_str}', descending=True).to_numpy()
        # Cut off r-squared column:
        arr = arr[:, :-1]
        picked_coeffs = pick_same_decrease(best_combo[p-1], arr)
        best_combo[p] = list(picked_coeffs)
        
    # Find the combos where values should be more than or same as the
    # starting values:
    for p in range(q-1, -1, -1):
        p_str = list(new_combo_100_df_dict.keys())[p]
        arr = new_combo_100_df_dict[p_str].sort(f'r2_{p_str}', descending=True).to_numpy()
        # Cut off r-squared column:
        arr = arr[:, :-1]
        picked_coeffs = pick_same_increase(best_combo[p+1], arr)
        best_combo[p] = list(picked_coeffs)
    best_picked_combos.append(sum(best_combo, []))

In [47]:
for best_combo in best_picked_combos:
    print(np.array(best_combo).reshape(5, 5))
    print('')

[[0.0004 0.006  0.006  0.006  0.014 ]
 [0.0004 0.004  0.004  0.004  0.014 ]
 [0.0003 0.003  0.003  0.004  0.014 ]
 [0.0003 0.003  0.003  0.003  0.013 ]
 [0.0002 0.003  0.003  0.003  0.013 ]]

[[0.0004 0.006  0.006  0.006  0.014 ]
 [0.0004 0.004  0.004  0.004  0.014 ]
 [0.0003 0.003  0.003  0.004  0.014 ]
 [0.0003 0.003  0.003  0.003  0.013 ]
 [0.0002 0.003  0.003  0.003  0.013 ]]

[[0.0004 0.005  0.005  0.007  0.015 ]
 [0.0004 0.003  0.003  0.007  0.014 ]
 [0.0003 0.002  0.002  0.007  0.014 ]
 [0.0003 0.002  0.002  0.006  0.013 ]
 [0.0002 0.002  0.002  0.005  0.013 ]]

[[0.0004 0.005  0.005  0.007  0.015 ]
 [0.0004 0.003  0.003  0.007  0.014 ]
 [0.0003 0.002  0.002  0.007  0.014 ]
 [0.0003 0.002  0.002  0.006  0.013 ]
 [0.0002 0.002  0.002  0.005  0.013 ]]

[[0.0004 0.006  0.006  0.006  0.014 ]
 [0.0004 0.003  0.004  0.006  0.014 ]
 [0.0003 0.003  0.003  0.005  0.014 ]
 [0.0002 0.003  0.003  0.005  0.013 ]
 [0.0001 0.003  0.003  0.004  0.013 ]]



Recalculate total r-squared:

In [48]:
for best_combo in best_picked_combos:
    list_r2 = calculate_many_rsquared(best_combo, x_lists, coeffs_ssnap, admissions_lists)
    # R2 for each quantile in turn, then R2 of all.
    
    print(list_r2)
    print('')

[0.5104249832646602, 0.5843986187150267, 0.6458026603158447, 0.6228825429469422, 0.6339766959485644, 0.6035739525065444]

[0.5104249832646602, 0.5843986187150267, 0.6458026603158447, 0.6228825429469422, 0.6339766959485644, 0.6035739525065444]

[0.5096507800694758, 0.5836669611382654, 0.6466793454130462, 0.6246806345257032, 0.6357727815367769, 0.6041890754801105]

[0.5096507800694758, 0.5836669611382654, 0.6466793454130462, 0.6246806345257032, 0.6357727815367769, 0.6041890754801105]

[0.5104249832646602, 0.5838457808004065, 0.6452648379405788, 0.6228492665504315, 0.6367176740638525, 0.6038615918298467]

